# UNHCR Global Displacement 2015–2025

**Companion notebook for the Medium article:**  
[How To Visualize 64 Million Internally Displaced People With Python And UNHCR Data](https://medium.com/@loewenj700)

Three charts built with Python + Plotly from UNHCR Refugee Data Finder data (global aggregates, 2015–2025).

**Dataset:** `unhcr_global.csv` — fetched from the [UNHCR Refugee Data Finder API](https://api.unhcr.org/population/v1/population/)  
**Author:** John Loewen | [Medium](https://medium.com/@loewenj700) | [GitHub](https://github.com/loewenj700)

---

## Setup

You need `pandas` and `plotly`. Install with:

```
pip install pandas plotly kaleido
```

`kaleido` is optional — only needed if you want to save PNG files.

In [ ]:
import pandas as pd
import plotly.graph_objects as go

## The Dataset

Load `unhcr_global.csv` — eleven rows, one per year 2015–2025. Columns we use:

- `year` — the year of the estimate
- `refugees` — people recognized as refugees under international law
- `asylum_seekers` — people with pending applications for refugee status
- `idps` — internally displaced people (displaced within their own country)
- `stateless` — people without a recognized national identity

In [ ]:
df = pd.read_csv('unhcr_global.csv')
print(df.columns.tolist())
print(df.head())

In [ ]:
# Quick shape check
print(df.shape)   # (11, 18)

---

## Chart 1: The Big Picture

A stacked area chart — one filled layer per displacement type, stacked so the total height shows the overall trend.

**Key finding:** IDPs (orange) account for 63% of all forcibly displaced people in 2025 and have grown the most since 2015.

In [ ]:
years  = list(df['year'])
idps   = [v / 1e6 for v in df['idps']]
asys   = [v / 1e6 for v in df['asylum_seekers']]
refs   = [v / 1e6 for v in df['refugees']]

IDP_COLOR = '#E07B39'
REF_COLOR = '#5B9BD5'
ASY_COLOR = '#6BAD72'

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=years, y=idps,
    fill='tozeroy', mode='lines',
    name='Internally Displaced (IDPs)',
    line=dict(color=IDP_COLOR, width=2.5),
    fillcolor='rgba(224,123,57,0.6)',
    stackgroup='one'
))
fig.add_trace(go.Scatter(
    x=years, y=asys,
    fill='tonexty', mode='lines',
    name='Asylum Seekers',
    line=dict(color=ASY_COLOR, width=2.5),
    fillcolor='rgba(107,173,114,0.55)',
    stackgroup='one'
))
fig.add_trace(go.Scatter(
    x=years, y=refs,
    fill='tonexty', mode='lines',
    name='Refugees',
    line=dict(color=REF_COLOR, width=2.5),
    fillcolor='rgba(91,155,213,0.55)',
    stackgroup='one'
))

fig.update_layout(
    title='Internal displacement is driving the global surge',
    xaxis_title='',
    yaxis_title='People displaced (millions)',
    plot_bgcolor='white'
)
fig.show()

---

## Chart 2: Five Years Of Change

A slope chart — two points per line, 2020 on the left and 2025 on the right — isolates the shift over the last five years.

**Key finding:** Asylum seekers more than doubled (4.2M → 9.0M). IDPs grew by 32% (48.6M → 64.2M).

In [ ]:
ref_2020 = df.loc[df['year'] == 2020, 'refugees'].values[0] / 1e6
asy_2020 = df.loc[df['year'] == 2020, 'asylum_seekers'].values[0] / 1e6
idp_2020 = df.loc[df['year'] == 2020, 'idps'].values[0] / 1e6

ref_2025 = df.loc[df['year'] == 2025, 'refugees'].values[0] / 1e6
asy_2025 = df.loc[df['year'] == 2025, 'asylum_seekers'].values[0] / 1e6
idp_2025 = df.loc[df['year'] == 2025, 'idps'].values[0] / 1e6

categories  = ['Refugees', 'Asylum Seekers', 'Internally Displaced (IDPs)']
vals_2020   = [ref_2020, asy_2020, idp_2020]
vals_2025   = [ref_2025, asy_2025, idp_2025]
colors      = ['#5B9BD5', '#6BAD72', '#E07B39']

fig = go.Figure()

for i in range(len(categories)):
    fig.add_trace(go.Scatter(
        x=[2020, 2025],
        y=[vals_2020[i], vals_2025[i]],
        mode='lines+markers',
        line=dict(color=colors[i], width=2 if i < 2 else 3.5),
        marker=dict(size=10, color=colors[i]),
        name=categories[i]
    ))

fig.update_layout(
    title='Asylum seekers more than doubled; IDPs surged by 32%',
    xaxis=dict(tickvals=[2020, 2025]),
    yaxis_title='Millions of people',
    plot_bgcolor='white'
)
fig.show()

---

## Chart 3: The Decade In Three Numbers

A bar chart of percentage change from 2015 to 2025 — makes all three displacement types comparable on the same scale.

**Key finding:** Asylum seekers grew 179% over the decade. Even on the elevated 2015 baseline, all three types have surged dramatically.

In [ ]:
ref_2015 = df.loc[df['year'] == 2015, 'refugees'].values[0]
asy_2015 = df.loc[df['year'] == 2015, 'asylum_seekers'].values[0]
idp_2015 = df.loc[df['year'] == 2015, 'idps'].values[0]

pct_ref = (ref_2025 * 1e6 - ref_2015) / ref_2015 * 100
pct_asy = (asy_2025 * 1e6 - asy_2015) / asy_2015 * 100
pct_idp = (idp_2025 * 1e6 - idp_2015) / idp_2015 * 100

categories  = ['Refugees', 'Asylum Seekers', 'Internally Displaced\n(IDPs)']
pcts        = [pct_ref, pct_asy, pct_idp]

fig = go.Figure()

for cat, pct, col in zip(categories, pcts, ['#5B9BD5', '#6BAD72', '#E07B39']):
    fig.add_trace(go.Bar(
        x=[cat], y=[pct],
        marker_color=col,
        text=[f'+{pct:.0f}%'],
        textposition='outside'
    ))

fig.update_layout(
    title='IDPs rose 71% since 2015 — asylum seekers nearly tripled',
    yaxis_title='Percentage increase (%)',
    plot_bgcolor='white',
    showlegend=False
)
fig.show()

---

**Source:** UNHCR Refugee Data Finder (2026). Global annual totals, 2015–2025.  
**Article:** [How To Visualize 64 Million Internally Displaced People With Python And UNHCR Data](https://medium.com/@loewenj700)  
**Author:** John Loewen — [Medium](https://medium.com/@loewenj700) | [Substack](https://johnloewen.substack.com/)